# CNN Architectures — Exercises

Companion to the note [CNN Architectures](CNN%20Architectures.md).

Practise why residual connections work, the arithmetic of output sizes, parameters and FLOPs (VGG, ResNet bottlenecks, depthwise separable convolutions, EfficientNet scaling, ViT tokens), then implement a shape tracer for VGG-16, residual versus plain gradient flow, 2D convolution, depthwise separable convolution, ViT patch embedding and batch normalisation from scratch in NumPy.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.

In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import softmax as sp_softmax
from scipy.signal import correlate2d
rng = np.random.default_rng(4)

def num_grad(f, x, eps=1e-6):
    """Central finite-difference gradient of a scalar function f() w.r.t. array x (modified in place, restored)."""
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

## Part A · Concepts

### Exercise 1 · Why residual connections work
*🧠 Concept · ★★☆*

1. Write the gradient $\partial L/\partial x$ of a residual block $y=x+F(x)$ and explain the role of the identity term.
2. Before ResNet a 56-layer plain network had *higher training error* than a 20-layer one. Why is that not overfitting, and what does it tell you about the problem?
3. What is the easiest function for a residual block to represent, and why is that useful?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Differentiate $y=x+F(x)$ with the chain rule. Overfitting shows up as low *training* error but high test error.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\big(I+\frac{\partial F}{\partial x}\big)$: the $I$ lets the gradient flow unchanged to earlier layers, even when $\partial F/\partial x$ is tiny.
2. The deeper net is worse **on the training set**, so it is an *optimisation* (degradation) problem: stacks of non-linear layers have difficulty even learning the identity mapping that a deeper model could in principle use.
3. $F(x)=0$, i.e. the identity. Extra blocks can then do no harm, which makes very deep networks trainable (100+ layers).

</details>

### Exercise 2 · Match the idea to the architecture
*🧠 Concept · ★☆☆*

For each model give the **one key idea** and the problem it addressed: AlexNet, VGG, GoogLeNet/Inception, DenseNet, MobileNet, EfficientNet, ViT, ConvNeXt.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Re-read the table of milestones in the note, but try from memory first.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Model | Key idea | Problem addressed |
|---|---|---|
| AlexNet | ReLU, dropout, GPUs, augmentation | made deep CNNs practical on ImageNet |
| VGG | stacks of $3\times3$ convs | simple, uniform design; depth helps |
| Inception | parallel branches + $1\times1$ bottlenecks | accuracy per FLOP |
| DenseNet | each layer sees all earlier feature maps | feature reuse, fewer parameters |
| MobileNet | depthwise separable convolutions | run on phones |
| EfficientNet | compound scaling of depth, width, resolution | balanced use of compute |
| ViT | image patches as tokens + Transformer | global receptive field from layer 1, scales with data |
| ConvNeXt | ResNet modernised with ViT design choices | showed the CNN/ViT gap was mostly the training recipe |

</details>

### Exercise 3 · Inductive bias, data and transfer learning
*🧠 Concept · ★★☆*

1. Which inductive biases does a CNN have that a ViT lacks? What is the consequence for the amount of training data?
2. You have 300 labelled images per class. Describe the transfer-learning recipe of the note (4 steps) and say why it beats training from scratch.
3. Why must you reuse the backbone's input normalisation and (roughly) its input resolution?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Locality and weight sharing. Think of what the pretrained filters already encode.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. CNNs bake in **locality** (small neighbourhoods) and **translation equivariance** (weight sharing); ViT has to learn such structure from data, so it needs more data, stronger augmentation or distillation (DeiT) — but it scales better with huge datasets.
2. Load an ImageNet-pretrained backbone → replace the head for your classes → train only the head with the backbone frozen → unfreeze and fine-tune everything with a small learning rate.
   The pretrained filters (edges, textures, parts) are already good features, so only a few parameters must be learned from the small dataset.
3. The pretrained filters expect inputs with the same statistics (mean/std per channel) and object scales they were trained on; otherwise the features are shifted and degrade.

</details>

## Part B · By hand

### Exercise 4 · Output size and parameters of the ResNet stem
*✍️ By hand · ★☆☆*

Input $224\times224\times3$. The stem is a conv with 64 filters of size $7\times7$, stride 2, padding 3, followed by a $3\times3$ max-pool with stride 2 and padding 1.
Compute the spatial size after the conv, the number of parameters of the conv (with biases), and the spatial size after the pool.

In [ ]:
size_conv = None
params_conv = None
size_pool = None
_o = lambda W, F, P, S: (W - F + 2 * P) // S + 1
check('size after conv', size_conv, _o(224, 7, 3, 2)); check('conv parameters', params_conv, (7 * 7 * 3 + 1) * 64)
check('size after pool', size_pool, _o(_o(224, 7, 3, 2), 3, 1, 2))

<details>
<summary><b>💡 Hint</b></summary>

Size $=\lfloor(W-F+2P)/S\rfloor+1$; parameters $=(F\cdot F\cdot C_{in}+1)C_{out}$. Pooling has no parameters.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Conv: $\lfloor(224-7+6)/2\rfloor+1=\lfloor111.5\rfloor+1=112$, giving $112\times112\times64$; parameters $(7\cdot7\cdot3+1)\cdot64=9{,}472$.
Pool: $\lfloor(112-3+2)/2\rfloor+1=55+1=56$, giving $56\times56\times64$.

```python
size_conv = 112
params_conv = 9472
size_pool = 56
```

</details>

### Exercise 5 · Stacked 3×3 convolutions versus one 7×7
*✍️ By hand · ★☆☆*

Let all layers have $C=64$ input and output channels (ignore biases). Compute the parameters of (a) one $7\times7$ conv and (b) three stacked $3\times3$ convs. What receptive field does the stack of three $3\times3$ (stride 1) see, and by what fraction is the stack cheaper?

In [ ]:
p_7x7 = None
p_3x3_stack = None
rf_stack = None
saving = None
check('7x7 parameters', p_7x7, 49 * 64**2); check('3x(3x3) parameters', p_3x3_stack, 27 * 64**2)
check('receptive field', rf_stack, 7); check('saving', saving, 1 - 27 / 49, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Parameters of one conv: $F^2C^2$. Each extra $3\times3$ layer adds 2 to the receptive field.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $49\cdot64^2=200{,}704$. (b) $3\cdot9\cdot64^2=110{,}592$. Three stacked $3\times3$ convs have receptive field $1+3\cdot2=7$, the same as one $7\times7$, but use $27/49$ of the parameters
($-44.9\%$) and add two more non-linearities. This is the VGG argument.

```python
p_7x7 = 200704
p_3x3_stack = 110592
rf_stack = 7
saving = 1 - 27 / 49
```

</details>

### Exercise 6 · ResNet bottleneck block
*✍️ By hand · ★★☆*

Compare the parameters (no biases, no BN) of the bottleneck block $1\times1$ ($256\to64$) → $3\times3$ ($64\to64$) → $1\times1$ ($64\to256$) with two plain $3\times3$ convs at 256 channels.
How many times cheaper is the bottleneck?

In [ ]:
p_bottleneck = None
p_plain = None
ratio = None
check('bottleneck parameters', p_bottleneck, 256*64 + 9*64*64 + 64*256); check('two 3x3 convs', p_plain, 2 * 9 * 256**2)
check('ratio', ratio, 2 * 9 * 256**2 / (256*64 + 9*64*64 + 64*256), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Bottleneck: $256\cdot64+9\cdot64^2+64\cdot256$. Plain: $2\cdot9\cdot256^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Bottleneck: $16{,}384+36{,}864+16{,}384=69{,}632$. Plain: $2\cdot9\cdot65{,}536=1{,}179{,}648$. Ratio $\approx16.9\times$.
The cheap $1\times1$ convs shrink the channel dimension so that the expensive $3\times3$ conv works on only 64 channels.

```python
p_bottleneck = 69632
p_plain = 1179648
ratio = 1179648 / 69632
```

</details>

### Exercise 7 · Depthwise separable convolution
*✍️ By hand · ★★☆*

A layer maps $56\times56\times128$ to $56\times56\times128$ with a $3\times3$ filter (stride 1, "same" padding). Count the multiplications of (a) a standard conv, (b) a depthwise conv followed by a pointwise $1\times1$ conv,
and the ratio (b)/(a). Compare with the formula $\frac1{C_{out}}+\frac1{F^2}$.

In [ ]:
mult_std = None
mult_sep = None
ratio = None
check('standard multiplications', mult_std, 462422016); check('separable multiplications', mult_sep, 54992896)
check('ratio = 1/C_out + 1/F^2', ratio, 1 / 128 + 1 / 9, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Per output pixel: standard $F^2C_{in}C_{out}$; depthwise $F^2C_{in}$ plus pointwise $C_{in}C_{out}$. There are $56\cdot56=3136$ pixels.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $9\cdot128\cdot128\cdot3136=462{,}422{,}016$. (b) $(9\cdot128+128\cdot128)\cdot3136=17{,}536\cdot3136=54{,}992{,}896$. Ratio $=0.1189=\frac1{128}+\frac19$, so about $8.4\times$ fewer multiplications.
For large $C_{out}$ the ratio tends to $1/F^2=1/9$.

```python
mult_std = 9 * 128 * 128 * 56 * 56
mult_sep = (9 * 128 + 128 * 128) * 56 * 56
ratio = mult_sep / mult_std
```

</details>

### Exercise 8 · EfficientNet compound scaling
*✍️ By hand · ★★☆*

EfficientNet scales depth by $\alpha^\phi$, width by $\beta^\phi$ and resolution by $\gamma^\phi$ with $\alpha=1.2,\ \beta=1.1,\ \gamma=1.15$ (chosen so that $\alpha\beta^2\gamma^2\approx2$).
Compute $\alpha\beta^2\gamma^2$, and for $\phi=3$: the FLOP multiplier $(\alpha\beta^2\gamma^2)^\phi$, the width multiplier and the input resolution when the base is $224$ px.

In [ ]:
base = None
flops_mult = None
width_mult = None
res_px = None
check('alpha*beta^2*gamma^2', base, 1.920, tol=1e-3); check('FLOP multiplier', flops_mult, 7.08, tol=0.01)
check('width multiplier', width_mult, 1.331, tol=1e-3); check('resolution (px)', res_px, 340.7, tol=0.1)

<details>
<summary><b>💡 Hint</b></summary>

FLOPs scale with depth, width squared and resolution squared.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\alpha\beta^2\gamma^2=1.2\cdot1.21\cdot1.3225=1.920$. For $\phi=3$: FLOPs $\times1.920^3\approx7.08$; width $\times1.1^3=1.331$; resolution $224\cdot1.15^3=224\cdot1.521\approx341$ px.
Scaling only one dimension (just depth, or just resolution) saturates quickly; scaling all three together spends the compute where it helps most.

```python
base = 1.2 * 1.1**2 * 1.15**2
flops_mult = base**3
width_mult = 1.1**3
res_px = 224 * 1.15**3
```

</details>

### Exercise 9 · Vision Transformer: tokens and cost
*✍️ By hand · ★★☆*

A ViT with patch size $P=16$ and width $d=768$ gets a $224\times224$ RGB image.
(a) How many patch tokens, and how many with the [CLS] token? (b) Parameters of the patch embedding (linear projection **with** bias)? (c) For a $384\times384$ image, how many tokens (with [CLS]) and by what factor does the self-attention score matrix per head grow compared with $224$?

In [ ]:
n_patches = None
n_tokens = None
embed_params = None
n_tokens_384 = None
score_growth = None
check('patches', n_patches, 196); check('tokens incl. [CLS]', n_tokens, 197); check('embedding parameters', embed_params, 16 * 16 * 3 * 768 + 768)
check('tokens at 384 px', n_tokens_384, 577); check('score-matrix growth', score_growth, 577**2 / 197**2, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Tokens $=(H/P)^2$. A flattened patch has $P\cdot P\cdot3$ numbers. The score matrix has (tokens)$^2$ entries.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $(224/16)^2=196$ patches, $197$ with [CLS]. (b) $(16\cdot16\cdot3)\cdot768+768=590{,}592$. (c) $(384/16)^2=576$, so $577$ tokens; score matrix $577^2=332{,}929$ vs $197^2=38{,}809$: a factor $\approx8.6$ for $1.71\times$ more pixels,
because attention is quadratic in the number of tokens.

```python
n_patches = 196
n_tokens = 197
embed_params = 590592
n_tokens_384 = 577
score_growth = 577**2 / 197**2
```

</details>

## Part C · Code from scratch

### Exercise 10 · Trace a network: VGG-16 shapes and parameters
*💻 Code · ★☆☆*

Implement `trace(cfg, H, C)` that walks through a list of layers and returns `(H, C, n_params)`: an integer in `cfg` is a $3\times3$ conv (stride 1, padding 1, with bias) with that many filters, `'M'` is a $2\times2$ max-pool with stride 2.
Run it on VGG-16 (13 convs). Then compute the parameters of the first fully connected layer ($\text{flatten}\to4096$, with bias) and what share of the total it makes up (conv + 3 FC layers $4096,4096,1000$).

In [ ]:
def trace(cfg, H, C):
    return None

fc1_params = None
share = None
vgg16 = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M']
res = trace(vgg16, 224, 3)
check('final shape (7, 512)', None if res is None else res[:2], (7, 512)); check('conv parameters', None if res is None else res[2], 14714688)
check('first FC layer', fc1_params, 7 * 7 * 512 * 4096 + 4096); check('its share of all parameters', share, 0.7428, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Use the output-size formula for the conv (stays the same with $3\times3$, $P=1$, $S=1$) and halve $H$ for pooling. Keep track of the current channel count $C$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The convs of VGG-16 have $14{,}714{,}688$ parameters and end at $7\times7\times512=25{,}088$ values. The first FC layer then has $25{,}088\cdot4096+4096=102{,}764{,}544$ parameters, more than all convs together:
with FC layers of $4096,4096,1000$ the total is $138{,}357{,}544$, of which about $74\%$ sit in the first FC layer. This is why later networks replace the FC head by global average pooling.

```python
def trace(cfg, H, C):
    total = 0
    for layer in cfg:
        if layer == 'M':
            H //= 2
        else:
            total += (3 * 3 * C + 1) * layer
            C = layer
    return H, C, total

fc1_params = 7 * 7 * 512 * 4096 + 4096
share = fc1_params / (14714688 + fc1_params + (4096 * 4096 + 4096) + (4096 * 1000 + 1000))
```

</details>

### Exercise 11 · Residual connections and vanishing gradients
*💻 Code · ★★☆*

Implement `input_grad(x, Ws, residual)`: the gradient of $L=\sum(\text{output})$ w.r.t. the input of a stack of layers $h\leftarrow\mathrm{relu}(Wh)$ (plain) or $h\leftarrow h+\mathrm{relu}(Wh)$ (residual), by manual backpropagation.
Verify it with finite differences on a 3-layer stack, then compare the gradient norm at the input of a **50-layer** plain and residual stack with the same weights.

In [ ]:
def input_grad(x, Ws, residual):
    return None
d_r = 32
Ws3 = [rng.normal(0, 0.3 / np.sqrt(d_r), size=(d_r, d_r)) for _ in range(3)]
x_r = rng.normal(size=d_r)
def _fwd(x, Ws, residual):
    h = x
    for W in Ws:
        h = h + np.maximum(0, W @ h) if residual else np.maximum(0, W @ h)
    return h.sum()
for res_flag in (False, True):
    check(f'gradient vs finite differences (residual={res_flag})', input_grad(x_r, Ws3, res_flag), num_grad(lambda: _fwd(x_r, Ws3, res_flag), x_r), tol=1e-5)
Ws50 = [rng.normal(0, 0.3 / np.sqrt(d_r), size=(d_r, d_r)) for _ in range(50)]
g_plain, g_res = input_grad(x_r, Ws50, False), input_grad(x_r, Ws50, True)
check('plain, 50 layers: gradient has vanished (norm < 1e-6)', None if g_plain is None else np.linalg.norm(g_plain) < 1e-6, True)
check('residual, 50 layers: gradient survives (norm > 1)', None if g_res is None else np.linalg.norm(g_res) > 1, True)

<details>
<summary><b>💡 Hint</b></summary>

Forward: store the pre-activations $z=Wh$. Backward through a layer: $g_a=g\odot(z>0)$, then $g_{in}=W^\top g_a$ (plain) or $g+W^\top g_a$ (residual).

</details>

<details>
<summary><b>✅ Solution</b></summary>

In the plain stack every layer multiplies the gradient by $W^\top D$ (with $D$ the ReLU mask), whose norm is below 1 for these weights, so after 50 layers it has vanished. In the residual stack the identity path
contributes $g$ unchanged at each layer, so the gradient reaching the input stays of order one. This is the $I+\partial F/\partial x$ of Exercise 1 in action.

```python
def input_grad(x, Ws, residual):
    h, pre = x, []
    for W in Ws:
        z = W @ h
        pre.append(z)
        h = h + np.maximum(0, z) if residual else np.maximum(0, z)
    g = np.ones_like(h)
    for W, z in zip(reversed(Ws), reversed(pre)):
        ga = g * (z > 0)
        g = g + W.T @ ga if residual else W.T @ ga
    return g
```

</details>

### Exercise 12 · 2D convolution from scratch
*💻 Code · ★★☆*

Implement `conv2d(X, Wt, stride=1, pad=0)` for `X` of shape `(H, W, C_in)` and filters `Wt` of shape `(F, F, C_in, C_out)` (cross-correlation, as in deep learning libraries). Return `(H_out, W_out, C_out)`.
Check the shape against the output-size formula and the values against `scipy.signal.correlate2d`.

In [ ]:
def conv2d(X, Wt, stride=1, pad=0):
    return None
def _conv_ref(X, Wt, stride, pad):
    Xp = np.pad(X, ((pad, pad), (pad, pad), (0, 0)))
    out = np.stack([sum(correlate2d(Xp[:, :, c], Wt[:, :, c, o], mode='valid') for c in range(X.shape[2])) for o in range(Wt.shape[3])], axis=-1)
    return out[::stride, ::stride]
X_c = rng.normal(size=(11, 9, 3)); W_c = rng.normal(size=(3, 3, 3, 4))
for stride, pad in [(1, 0), (1, 1), (2, 1)]:
    y = conv2d(X_c, W_c, stride, pad)
    check(f'stride={stride}, pad={pad}', y, _conv_ref(X_c, W_c, stride, pad))
y = conv2d(X_c, W_c, 2, 1)
check('output shape follows the formula', None if y is None else y.shape, ((11 - 3 + 2) // 2 + 1, (9 - 3 + 2) // 2 + 1, 4))

<details>
<summary><b>💡 Hint</b></summary>

Zero-pad with `np.pad`. For every output position take the patch `Xp[i*s:i*s+F, j*s:j*s+F, :]` and contract it with the filters: `np.tensordot(patch, Wt, axes=([0, 1, 2], [0, 1, 2]))` gives all $C_{out}$ values.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each output pixel is the dot product of a $F\times F\times C_{in}$ patch with a filter, so a conv layer is a sliding, weight-shared matrix multiplication; this is also how `im2col` implements it on GPUs.
The output size is $\lfloor(W-F+2P)/S\rfloor+1$ in each direction.

```python
def conv2d(X, Wt, stride=1, pad=0):
    F = Wt.shape[0]
    Xp = np.pad(X, ((pad, pad), (pad, pad), (0, 0)))
    Ho = (Xp.shape[0] - F) // stride + 1
    Wo = (Xp.shape[1] - F) // stride + 1
    out = np.zeros((Ho, Wo, Wt.shape[3]))
    for i in range(Ho):
        for j in range(Wo):
            patch = Xp[i*stride:i*stride+F, j*stride:j*stride+F, :]
            out[i, j] = np.tensordot(patch, Wt, axes=([0, 1, 2], [0, 1, 2]))
    return out
```

</details>

### Exercise 13 · Depthwise separable convolution in code
*💻 Code · ★★☆*

Implement `depthwise_separable(X, Wd, Wp)` with "same" padding and stride 1: a depthwise conv (`Wd` has shape `(F, F, C)`, one filter per channel) followed by a pointwise $1\times1$ conv (`Wp` has shape `(C, C_out)`).
Show that it equals a standard convolution whose filters are the rank-one combination $W[u,v,c,o]=W_d[u,v,c]\,W_p[c,o]$, and count the parameters for $C=16$, $C_{out}=32$, $F=3$.

In [ ]:
def depthwise_separable(X, Wd, Wp):
    return None

p_sep = None
p_std = None
X_s = rng.normal(size=(10, 8, 16)); Wd_s = rng.normal(size=(3, 3, 16)); Wp_s = rng.normal(size=(16, 32))
_W = Wd_s[:, :, :, None] * Wp_s[None, None, :, :]
_ref = np.stack([sum(correlate2d(np.pad(X_s, ((1, 1), (1, 1), (0, 0)))[:, :, c], _W[:, :, c, o], mode='valid') for c in range(16)) for o in range(32)], axis=-1)
check('equals the standard conv with rank-one filters', depthwise_separable(X_s, Wd_s, Wp_s), _ref)
check('separable parameters', p_sep, 656); check('standard parameters', p_std, 4608)

<details>
<summary><b>💡 Hint</b></summary>

Depthwise: `out[i, j, c] = sum_{u,v} Xp[i+u, j+v, c] * Wd[u, v, c]`, loop over $u,v$ and use whole-image slices. Pointwise: `np.einsum('ijc,co->ijo', D, Wp)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Depthwise filters never mix channels; the pointwise conv mixes them at every pixel. The pair is a standard convolution restricted to *separable* filters (spatial part $\otimes$ channel part), hence far fewer parameters:
$F^2C+C\,C_{out}=144+512=656$ instead of $F^2C\,C_{out}=4{,}608$ ($7\times$ fewer).

```python
def depthwise_separable(X, Wd, Wp):
    F = Wd.shape[0]
    Xp = np.pad(X, ((F // 2, F // 2), (F // 2, F // 2), (0, 0)))
    D = np.zeros_like(X, dtype=float)
    for u in range(F):
        for v in range(F):
            D += Xp[u:u + X.shape[0], v:v + X.shape[1], :] * Wd[u, v]
    return np.einsum('ijc,co->ijo', D, Wp)

p_sep = 3 * 3 * 16 + 16 * 32
p_std = 3 * 3 * 16 * 32
```

</details>

### Exercise 14 · ViT patch embedding
*💻 Code · ★★☆*

Implement `patchify(img, P)` which cuts an image `(H, W, C)` into non-overlapping $P\times P$ patches in row-major order and flattens each to a vector, giving `(N, P*P*C)`, plus its inverse `unpatchify`.
Then build the ViT input tokens: `vit_tokens(img, P, We, be, cls, pos)` = `concat([cls], patches @ We + be) + pos`, shape `(N+1, d)`.

In [ ]:
def patchify(img, P):
    return None

def unpatchify(patches, H, W, C, P):
    return None

def vit_tokens(img, P, We, be, cls, pos):
    return None
img = rng.normal(size=(224, 224, 3)); P_v, d_v = 16, 64
pt = patchify(img, P_v)
check('196 patches of 16*16*3 numbers', None if pt is None else pt.shape, (196, 768))
check('patch 0 is the top-left block', None if pt is None else pt[0], img[:16, :16].ravel())
check('patch 1 is the next block to the right', None if pt is None else pt[1], img[:16, 16:32].ravel())
check('patch 14 starts the second row of patches', None if pt is None else pt[14], img[16:32, :16].ravel())
check('unpatchify is the inverse', None if pt is None else unpatchify(pt, 224, 224, 3, P_v), img)
We_v, be_v, cls_v, pos_v = rng.normal(size=(768, d_v)), rng.normal(size=d_v), rng.normal(size=d_v), rng.normal(size=(197, d_v))
tok = vit_tokens(img, P_v, We_v, be_v, cls_v, pos_v)
check('token matrix is (197, d)', None if tok is None else tok.shape, (197, d_v))
check('row 0 = [CLS] + pos[0]', None if tok is None else tok[0], cls_v + pos_v[0])

<details>
<summary><b>💡 Hint</b></summary>

`img.reshape(H//P, P, W//P, P, C).transpose(0, 2, 1, 3, 4).reshape(N, -1)`. The inverse reverses the reshape/transpose.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Cutting into patches is only a reshape and transpose: no parameters. The linear projection (equivalently a conv with kernel $P$ and stride $P$) maps each flattened patch to $d$ dimensions; the learnable [CLS] token
and the position embeddings (without them ViT is permutation-equivariant just like the Transformer) are then added.

```python
def patchify(img, P):
    H, W, C = img.shape
    return img.reshape(H // P, P, W // P, P, C).transpose(0, 2, 1, 3, 4).reshape((H // P) * (W // P), -1)

def unpatchify(patches, H, W, C, P):
    return patches.reshape(H // P, W // P, P, P, C).transpose(0, 2, 1, 3, 4).reshape(H, W, C)

def vit_tokens(img, P, We, be, cls, pos):
    return np.vstack([cls, patchify(img, P) @ We + be]) + pos
```

</details>

### Exercise 15 · Batch normalisation for conv features
*💻 Code · ★★☆*

Implement `batchnorm_forward(X, gamma, beta, running, train, momentum=0.1, eps=1e-5)` for conv feature maps `X` of shape `(N, H, W, C)`: in training mode normalise each **channel** with the mean and variance over $(N,H,W)$, update
`running['mean']`, `running['var']` as $r\leftarrow(1-m)\,r+m\cdot\text{batch statistic}$; in evaluation mode use the running statistics instead. Output $\gamma\hat x+\beta$.

In [ ]:
def batchnorm_forward(X, gamma, beta, running, train, momentum=0.1, eps=1e-5):
    return None
Cn = 3; mean_true, std_true = np.array([3., -2., 0.5]), np.array([2., 0.5, 1.])
batch = lambda: rng.normal(mean_true, std_true, size=(8, 5, 5, Cn))
running = dict(mean=np.zeros(Cn), var=np.ones(Cn))
Xb = batch(); Y = batchnorm_forward(Xb, np.ones(Cn), np.zeros(Cn), running, train=True)
check('train: per-channel mean 0', None if Y is None else Y.mean(axis=(0, 1, 2)), np.zeros(Cn), tol=1e-6)
check('train: per-channel variance 1', None if Y is None else Y.var(axis=(0, 1, 2)), np.ones(Cn), tol=1e-3)
check('running mean moved 10% towards the batch mean', running['mean'] if Y is not None else None, 0.1 * Xb.mean(axis=(0, 1, 2)))
Y2 = batchnorm_forward(Xb, 2 * np.ones(Cn), np.ones(Cn), dict(mean=np.zeros(Cn), var=np.ones(Cn)), train=True)
check('gamma=2, beta=1: mean 1, std 2', None if Y2 is None else [Y2.mean(), Y2.std()], [1, 2], tol=1e-2)
for _ in range(100):
    batchnorm_forward(batch(), np.ones(Cn), np.zeros(Cn), running, train=True)
check('running mean converges to the true mean', None if Y is None else running['mean'], mean_true, tol=0.35)
x1 = batch()[:1]; y1 = batchnorm_forward(x1, np.ones(Cn), np.zeros(Cn), running, train=False)
check('eval: one image, uses running statistics', y1, (x1 - running['mean']) / np.sqrt(running['var'] + 1e-5))

<details>
<summary><b>💡 Hint</b></summary>

`X.mean(axis=(0, 1, 2))` gives one value per channel. Use `var` with the same axes (biased, i.e. divide by the count).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Statistics are computed per channel, over the batch **and** all spatial positions. At test time there may be a single image, so the layer uses the running averages collected during training: the output of one image then no longer depends on the other images in the batch,
unlike in training. This train/eval difference is a classical source of bugs (and one reason LayerNorm is preferred in Transformers).

```python
def batchnorm_forward(X, gamma, beta, running, train, momentum=0.1, eps=1e-5):
    if train:
        mu, var = X.mean(axis=(0, 1, 2)), X.var(axis=(0, 1, 2))
        running['mean'] = (1 - momentum) * running['mean'] + momentum * mu
        running['var'] = (1 - momentum) * running['var'] + momentum * var
    else:
        mu, var = running['mean'], running['var']
    return gamma * (X - mu) / np.sqrt(var + eps) + beta
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [CNN Architectures](CNN%20Architectures.md).*